# Fine-tuning a pretrained Transformer

124 million parameters, one epoch, and 93.7% on the problem where chapter 14 hit a 90% ceiling.

**Runs on:** GPU strongly recommended · downloads ~500 MB &nbsp;·&nbsp; **Slides:** [Chapter 15 — Language Models and the Transformer](../../../course-web-slides/ch15/index.html) &nbsp;·&nbsp; **Section:** 04 — Classification with a pretrained Transformer

---

## Causal and masked language models

| | Causal LM | Masked LM |
|---|---|---|
| Distribution | p(token \| **past**) | p(token \| **surrounding**) |
| Attention | causal-masked | bidirectional |
| Good at | **generating** | **representing** |
| Examples | GPT, Llama — chapter 16 | BERT, RoBERTa — here |

BERT masks about **15% of tokens** and predicts the originals. No labels are needed, which is what made the training data available.

## Loading RoBERTa

In [ ]:
import keras
import keras_hub

tokenizer = keras_hub.models.Tokenizer.from_preset("roberta_base_en")
backbone = keras_hub.models.Backbone.from_preset("roberta_base_en")

print(f"{backbone.count_params():,} parameters")
backbone.summary()

**RoBERTa used 160 GB of web text against BERT's 16 GB of Wikipedia** — same architecture, ten times the data, and an estimated few hundred thousand dollars of compute. Twelve encoder blocks, which is the stacking property we relied on in notebook 04.

## The tokenizer, and why subwords are mandatory here

In [ ]:
print(tokenizer("The quick brown fox"))
print()
print("vocabulary size:", tokenizer.vocabulary_size())
print()
for text in ["antidisestablishmentarianism", "keras", "COVID-19"]:
    ids = tokenizer(text)
    print(f"{text:32s} -> {len(ids)} tokens")

A 50,000-term vocabulary handles **any** word, including ones coined after training. Character-level would make sequences far too long (attention costs grow with the square of length); word-level would need a vocabulary covering millions of web documents.

## Packing, exactly as pretraining did

In [ ]:
import tensorflow as tf
from keras.utils import text_dataset_from_directory
import pathlib

base_dir = pathlib.Path("aclImdb")
batch_size = 16       # 512-token sequences through 124M parameters

train_ds = text_dataset_from_directory(base_dir / "train", batch_size=batch_size)
val_ds = text_dataset_from_directory(base_dir / "val", batch_size=batch_size)
test_ds = text_dataset_from_directory(base_dir / "test", batch_size=batch_size)

def preprocess(text, label):
    packer = keras_hub.layers.StartEndPacker(
        sequence_length=512,
        start_value=tokenizer.start_token_id,
        end_value=tokenizer.end_token_id,
        pad_value=tokenizer.pad_token_id,
        return_padding_mask=True,
    )
    token_ids, padding_mask = packer(tokenizer(text))
    return {"token_ids": token_ids, "padding_mask": padding_mask}, label

train_p = train_ds.map(preprocess)
val_p = val_ds.map(preprocess)
test_p = test_ds.map(preprocess)

x, y = next(iter(train_p))
print({k: v.shape for k, v in x.items()}, y.shape)

> ⚠️ **Match the pretraining token order.** RoBERTa expects `<s>`, content, `</s>`, then `<pad>`. Getting this wrong does not error — it just trains more slowly and scores worse.

## The classification head, and why token zero

In [ ]:
from keras import layers

inputs = backbone.input
x = backbone(inputs)
x = x[:, 0, :]                              # the first token's representation
x = layers.Dropout(0.1)(x)
x = layers.Dense(768, activation="relu")(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
classifier = keras.Model(inputs, outputs)
print(f"{classifier.count_params():,} parameters")

Mean or max pooling would also work. Taking the first token works slightly better, and the reason is attention: **the first position in the final encoder layer can attend to every other position**. Rather than pooling with something coarse, attention pools contextually.

## One epoch, at 5e-5

In [ ]:
classifier.compile(
    optimizer=keras.optimizers.Adam(5e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
classifier.fit(train_p, validation_data=val_p, epochs=1, verbose=2)
print("\ntest:", classifier.evaluate(test_p, verbose=0))

Expected output:

```
test: [0.168, 0.9366]
```

**93.7% after a single epoch**, against the 90% ceiling of chapter 14.

`5e-5` is roughly twenty times smaller than Adam's default. Same discipline as chapter 8's fine-tuning, for the same reason: large updates destroy representations that cost $300,000 to learn.

## What it cost

In [ ]:
print(f"{'model':38s} {'accuracy':>9s} {'cost':>26s}")
print("-" * 76)
rows = [("bigram bag-of-words (ch14)", 0.902, "seconds, CPU"),
        ("sequence model + embeddings (ch14)", 0.90, "minutes, one GPU"),
        ("RoBERTa base, 1 epoch", 0.937, "124M params, ~1 GPU-hour"),
        ("RoBERTa large, fine-tuned", 0.95, "300M params")]
for name, acc, cost in rows:
    print(f"{name:38s} {acc:>9.3f} {cost:>26s}")
print()
print("Three points of accuracy for roughly four orders of magnitude")
print("of compute. Whether that is worth it is chapter 18's question,")
print("and it does not have a universal answer.")

## Try the ablation that matters

In [ ]:
import numpy as np

# Freeze the backbone entirely: train only the head.
backbone.trainable = False
frozen = keras.Model(inputs, outputs)
frozen.compile(optimizer=keras.optimizers.Adam(1e-3),
               loss="binary_crossentropy", metrics=["accuracy"])
print("trainable parameters with the backbone frozen:",
      f"{sum(int(np.prod(w.shape)) for w in frozen.trainable_weights):,}")
print()
print("Train this and compare. Feature extraction alone usually gets")
print("most of the way; fine-tuning the backbone buys the last points --")
print("the same ordering as chapter 8's vision models.")

Chapter 16 takes this further with **LoRA**, which makes fine-tuning a billion-parameter model fit in 16 GB — a technique that only exists because full fine-tuning does not.

---

## What to take away

- Masked LMs represent text; causal LMs generate it. RoBERTa is the former.
- Match the pretraining tokenizer **and** its packing format.
- Take the first token's representation — attention has already pooled contextually.
- One epoch at 5e-5 gives 93.7%, for about four orders of magnitude more compute than a bigram model.